# 🚀 Huấn luyện Qwen-7B (QLoRA 4-bit) An Toàn VRAM trên Google Colab
Notebook này được tối ưu riêng biệt cho GPU **Tesla T4 (15.360 MiB)** trên Google Colab:
- **Quantization**: 4-bit NormalFloat (NF4) + Double Quantization (giảm 75% VRAM).
- **Precision**: Float16 (`torch_dtype=torch.float16`), vô hiệu hóa BFloat16 để tránh lỗi GradScaler trên kiến trúc Turing.
- **LoRA Config**: `r=8`, `lora_alpha=16`, target `["q_proj", "v_proj"]` (VRAM đỉnh chỉ ~7.8 GB, an toàn tuyệt đối trên 15 GB của Colab T4).
- **Bộ công cụ VRAM**: Tích hợp module kiểm tra Driver GPU qua nvidia-smi và xả rác bộ nhớ tức thì.


In [ ]:
# [Bước 1] Kiểm tra cấu hình GPU trên Colab (Tesla T4 15.360 MiB)
!nvidia-smi


In [ ]:
# [Bước 2] Cài đặt & cập nhật các thư viện Deep Learning tương thích GPU Colab
!pip install -U -q bitsandbytes transformers accelerate peft trl datasets
print("✅ Cài đặt thư viện hoàn tất!")


In [ ]:
# [Bước 3] Chuẩn bị tập dữ liệu dataset.jsonl
import os
from google.colab import files

if not os.path.exists("dataset.jsonl"):
    print("👉 Vui lòng chọn file 'dataset.jsonl' từ máy tính của bạn để tải lên:")
    uploaded = files.upload()
else:
    print("✅ File dataset.jsonl đã sẵn sàng trên Colab!")


In [ ]:
# [Bước 3.5] BỘ CÔNG CỤ QUẢN TRỊ & GIẢI PHÓNG VRAM TOÀN DIỆN TRÊN GOOGLE COLAB
import gc
import os
import torch
import subprocess

def xem_vram_thuc_te():
    """Đọc VRAM trực tiếp từ phần cứng GPU qua nvidia-smi (chuẩn xác 100% cấp độ Driver)."""
    try:
        res = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=memory.total,memory.used,memory.free,temperature.gpu", "--format=csv,nounits,noheader"],
            encoding="utf-8"
        ).strip().split(",")
        total, used, free, temp = [float(x.strip()) for x in res]
        return total / 1024, used / 1024, free / 1024, temp
    except Exception:
        if torch.cuda.is_available():
            total = torch.cuda.get_device_properties(0).total_memory / (1024**3)
            used = torch.cuda.memory_allocated(0) / (1024**3)
            return total, used, total - used, 0.0
        return 0, 0, 0, 0

def in_thanh_tien_trinh_vram(used, total, label=""):
    pct = (used / total) * 100 if total > 0 else 0
    bar_len = 25
    filled = int(round(bar_len * (pct / 100.0)))
    bar = "█" * filled + "░" * (bar_len - filled)
    color = "🟢" if pct < 60 else ("🟡" if pct < 85 else "🔴")
    print(f"{label:<12} [{bar}] {pct:5.1f}% | Đã dùng: {used:5.2f} GB / {total:5.2f} GB {color}")

def giai_phong_vram(verbose=True):
    """Thu gom toàn bộ biến mô hình, xả sạch bộ nhớ đệm CUDA và trả lại VRAM trống."""
    if not torch.cuda.is_available():
        print("⚠️ Không tìm thấy GPU CUDA trên Colab session này.")
        return

    gpu_name = torch.cuda.get_device_name(0)
    total_hw, used_before, free_before, temp = xem_vram_thuc_te()

    if verbose:
        print("=" * 70)
        print(f"📊 GIÁM SÁT VRAM GPU COLAB: {gpu_name} (Nhiệt độ: {temp:.0f}°C)")
        print("-" * 70)
        in_thanh_tien_trinh_vram(used_before, total_hw, label="Trước dọn:")

    # 1. Hủy tham chiếu các biến mô hình AI trong namespace global
    target_vars = ['model', 'trainer', 'base_model', 'peft_model', 'inputs', 'outputs', 'raw_dataset', 'formatted_dataset']
    deleted_count = 0
    for v in target_vars:
        if v in globals():
            del globals()[v]
            deleted_count += 1

    # 2. Ép Garbage Collection đa cấp
    gc.collect()

    # 3. Xả sạch bộ nhớ đệm PyTorch Caching Allocator & IPC
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    torch.cuda.reset_peak_memory_stats()

    # 4. Kiểm tra lại sau khi giải phóng
    total_hw, used_after, free_after, _ = xem_vram_thuc_te()
    freed = used_before - used_after

    if verbose:
        in_thanh_tien_trinh_vram(used_after, total_hw, label="Sau khi dọn:")
        print("-" * 70)
        if freed > 0.05:
            print(f"🧹 Đã thu hồi thành công {freed:.2f} GB VRAM (Đã dọn {deleted_count} biến mô hình tồn đọng)!")
        else:
            print("✨ VRAM đã ở trạng thái tối ưu, không có vùng nhớ rác bị khóa.")
        print(f"✅ VRAM KHẢ DỤNG HIỆN TẠI: {free_after:.2f} GB (Sẵn sàng nạp Qwen-7B)")
        print("=" * 70)

def reset_nhanh_colab():
    """Khởi động lại Python kernel trong 2 giây nếu VRAM bị crash/kẹt cứng (Giữ nguyên toàn bộ file)."""
    print("🔄 Đang khởi động lại Colab session... VRAM sẽ sạch 100% trong 2 giây!")
    os.kill(os.getpid(), 9)

# KÍCH HOẠT KIỂM TRA VÀ DỌN DẸP NGAY LẬP TỨC
giai_phong_vram()


In [ ]:
# [Bước 4] HUẤN LUYỆN MÔ HÌNH QWEN 7B (TỐI ƯU VRAM & TRIỆT TIÊU 100% LỖI GRADSCALER BFLOAT16)
import os
import gc
import json
import glob
import torch
import inspect

# 0. KHẮC PHỤC LỖI THIẾU BINARY BITSANDBYTES (CUDA 130/132 / TRITON.OPS) TRÊN COLAB
try:
    import site
    for sp in site.getsitepackages():
        cuda_libs = glob.glob(os.path.join(sp, "bitsandbytes", "libbitsandbytes_cuda*.so"))
        if cuda_libs:
            vers = [int(os.path.basename(f).replace("libbitsandbytes_cuda", "").replace(".so", "")) 
                    for f in cuda_libs if os.path.basename(f).replace("libbitsandbytes_cuda", "").replace(".so", "").isdigit()]
            if vers:
                best_ver = str(max(vers))
                os.environ["BNB_CUDA_VERSION"] = best_ver
                print(f"⚡ Đã kích hoạt GPU binary cho bitsandbytes: BNB_CUDA_VERSION = {best_ver}")
                break
except Exception:
    os.environ["BNB_CUDA_VERSION"] = "124"

# 1. Giải phóng bộ nhớ đệm
gc.collect()
torch.cuda.empty_cache()

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
import bitsandbytes as bnb

print(f"✅ BitsAndBytes đã nạp thành công với GPU (Phiên bản: {bnb.__version__})")

# 2. Cấu hình mô hình
MODEL_TIER = "7b"
MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct"
STEPS = 100
OUTPUT_DIR = "outputs"
FINAL_LORA_DIR = os.path.join(OUTPUT_DIR, "course-management-lora-adapter")

print(f"🚀 Đang nạp mô hình nền {MODEL_ID} (NF4 4-bit) vào GPU Tesla T4...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Nạp mô hình 4-bit
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
model.config.torch_dtype = torch.float16
model.config.use_cache = False

model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

# Cấu hình LoRA tối ưu VRAM: chỉ train q_proj và v_proj để VRAM chỉ chiếm ~7.8 GB / 15 GB
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)

# Ép toàn bộ tham số trainable sang float32 để ổn định gradients
for p in model.parameters():
    if p.requires_grad:
        p.data = p.data.to(torch.float32)

print("\n--- THAM SỐ HUẤN LUYỆN ĐÃ ĐƯỢC TỐI ƯU VRAM ---")
model.print_trainable_parameters()

# 3. Nạp và xử lý tập dữ liệu
print("\nĐang xử lý dataset.jsonl...")
raw_dataset = load_dataset("json", data_files="dataset.jsonl", split="train")

def format_chat(batch):
    formatted_texts = []
    for messages in batch["messages"]:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        formatted_texts.append(text)
    return {"text": formatted_texts}

formatted_dataset = raw_dataset.map(format_chat, batched=True)

# 4. CẤU HÌNH SIÊU THAM SỐ: TẮT GRADSCALER ĐỂ KHẮC PHỤC TRIỆT ĐỂ LỖI BFLOAT16
# Chú thích: Khi dùng QLoRA (trọng số trainable là float32), việc tắt fp16/bf16 ở Trainer
# sẽ vô hiệu hóa GradScaler của PyTorch -> Không còn gọi hàm _amp_foreach_non_finite_check_and_unscale_cuda!
sft_sig = inspect.signature(SFTConfig.__init__).parameters
sft_kwargs = {
    'output_dir': OUTPUT_DIR,
    'per_device_train_batch_size': 1,
    'gradient_accumulation_steps': 16,
    'learning_rate': 2e-4,
    'logging_steps': 5,
    'max_steps': STEPS,
    'save_strategy': 'no',
    'fp16': False,               # TẮT FP16 ĐỂ TẮT HOÀN TOÀN GRADSCALER
    'bf16': False,               # TẮT BF16 VÌ TESLA T4 KHÔNG HỖ TRỢ NATIVE
    'optim': 'paged_adamw_8bit',
    'gradient_checkpointing': True,
    'warmup_steps': 5,
    'lr_scheduler_type': 'cosine',
    'report_to': 'none'
}

# Tự động thích ứng: TRL mới dùng max_length, TRL cũ dùng max_seq_length
if 'max_length' in sft_sig:
    sft_kwargs['max_length'] = 512
elif 'max_seq_length' in sft_sig:
    sft_kwargs['max_seq_length'] = 512

if 'dataset_text_field' in sft_sig:
    sft_kwargs['dataset_text_field'] = 'text'

sft_args = SFTConfig(**sft_kwargs)

trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset,
    args=sft_args
)

print("\n" + "="*60)
print(f"🔥 BẮT ĐẦU HUẤN LUYỆN ({STEPS} steps, Micro-batch: 1, Accumulation: 16)... VRAM ~7.5 GB / 15 GB!")
print("="*60)
torch.cuda.empty_cache()
trainer.train()

# 5. Lưu trọng số LoRA Adapter
os.makedirs(FINAL_LORA_DIR, exist_ok=True)
trainer.model.save_pretrained(FINAL_LORA_DIR)
tokenizer.save_pretrained(FINAL_LORA_DIR)

with open(os.path.join(FINAL_LORA_DIR, "model_meta.json"), "w", encoding="utf-8") as f:
    json.dump({"base_model_id": MODEL_ID, "model_tier": MODEL_TIER}, f, indent=2)

print("\n" + "="*60)
print(f"🎉 HUẤN LUYỆN THÀNH CÔNG MÔ HÌNH {MODEL_ID}!")
print(f"LoRA Adapter đã được lưu tại: {FINAL_LORA_DIR}")
print("="*60)


In [ ]:
# [Bước 5] ĐÓNG GÓI VÀ TỰ ĐỘNG TẢI LORA ADAPTER VỀ MÁY TÍNH
from google.colab import files

!zip -r lora_adapter_colab.zip outputs/course-management-lora-adapter
print("\n👉 Trình duyệt đang tải file 'lora_adapter_colab.zip' về máy của bạn...")
files.download('lora_adapter_colab.zip')


In [ ]:
# [Bước 6] THỬ NGHIỆM ĐỐI THOẠI TRỰC TIẾP VỚI MÔ HÌNH VỪA TRAIN NGAY TRÊN COLAB
import torch

model.eval()

def hoi_dap_ai(cau_hoi):
    print(f"\n❓ Câu hỏi: {cau_hoi}")
    print("-" * 60)
    messages = [
        {"role": "system", "content": "Bạn là kỹ sư AI phần mềm cao cấp, chuyên gia kiến trúc Spring Boot 3, Java 21, Hibernate Generic DAO của dự án course-management-system."},
        {"role": "user", "content": cau_hoi}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1
        )
    tra_loi = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    print(f"🤖 Qwen AI Trả lời:\n{tra_loi}\n")

# Chạy thử câu hỏi về kiến trúc dự án:
hoi_dap_ai("Hãy viết phương thức findByStatus trong CourseDaoImpl theo đúng chuẩn Hibernate SessionFactory của dự án.")
